# BNP Paribas × Société Générale — full pairs-trading pipeline

This notebook downloads adjusted prices from 2020 through the latest available 2026 observation, estimates a dynamic Kalman hedge ratio, builds a rolling spread z-score, and runs the current `PairsBacktester` implementation.

The final panel compares the strategy with a simple 50/50 buy-and-hold benchmark made from the two normalized stocks. This is an exploratory backtest, not investment advice. Historical cointegration and simulated results do not imply future performance.


In [ ]:
from pathlib import Path
import sys

# Supports running Jupyter from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data import download_adjusted_closes
from src.analytics import kalman_hedge_ratio, dynamic_spread, rolling_zscore
from src.backtest import PairsBacktester


In [ ]:
# yfinance uses an exclusive end date; 2026-12-31 lets the request include
# all available observations through 2026, even when run before year-end.
prices = download_adjusted_closes(
    start="2020-01-01",
    end="2026-12-31",
    tickers=("BNP.PA", "GLE.PA"),
)
prices = prices.dropna().sort_index()
print(f"Rows: {len(prices):,}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail()


In [ ]:
# Dynamic hedge ratio and residual spread, estimated from adjusted price levels.
kalman_state = kalman_hedge_ratio(
    prices,
    dependent="BNP.PA",
    independent="GLE.PA",
)
spread = dynamic_spread(prices, kalman_state)
zscore = rolling_zscore(spread, window=60)
beta_t = kalman_state["hedge_ratio"]

ENTRY_Z = 1.5
EXIT_Z = 0.0
backtester = PairsBacktester(
    spread_zscore=zscore,
    prices=prices[["BNP.PA", "GLE.PA"]],
    hedge_ratio=beta_t,
    entry_threshold=ENTRY_Z,
    exit_threshold=EXIT_Z,
    transaction_fee=0.001,  # 10 bps per unit of turnover
)
summary, pnl = backtester.run()
pd.Series(summary, name="Strategy summary").to_frame()


In [ ]:
# Equal-weight buy-and-hold benchmark: 50% in each adjusted-price series.
normalized_prices = prices / prices.iloc[0] * 100.0
buy_hold = (normalized_prices / 100.0).mean(axis=1) - 1.0
buy_hold = buy_hold.reindex(pnl.index).ffill()

positions = pnl["position"].reindex(zscore.index).fillna(0.0)
previous_positions = positions.shift(1).fillna(0.0)
long_entries = (positions == 1.0) & (previous_positions == 0.0)
short_entries = (positions == -1.0) & (previous_positions == 0.0)
exits = (positions == 0.0) & (previous_positions != 0.0)

fig, axes = plt.subplots(
    3, 1, figsize=(15, 14), sharex=True,
    gridspec_kw={"height_ratios": [1.1, 1.0, 1.6]},
)

# 1) Normalized adjusted prices.
axes[0].plot(normalized_prices.index, normalized_prices["BNP.PA"], label="BNP Paribas (BNP.PA)")
axes[0].plot(normalized_prices.index, normalized_prices["GLE.PA"], label="Société Générale (GLE.PA)")
axes[0].set_title("1. Normalized adjusted prices (start = 100)")
axes[0].set_ylabel("Index level")
axes[0].legend(loc="upper left")
axes[0].grid(alpha=0.25)

# 2) Dynamic Kalman hedge ratio.
axes[1].plot(beta_t.index, beta_t, color="darkorange", label="Dynamic hedge ratio βₜ")
axes[1].set_title("2. Kalman dynamic hedge ratio")
axes[1].set_ylabel("βₜ")
axes[1].legend(loc="upper left")
axes[1].grid(alpha=0.25)

# 3) Spread z-score, entry/exit markers, and cumulative performance.
ax_z = axes[2]
ax_z.plot(zscore.index, zscore, color="steelblue", label="Spread z-score")
ax_z.axhline(ENTRY_Z, color="firebrick", linestyle="--", label=f"Short-spread entry (+{ENTRY_Z})")
ax_z.axhline(-ENTRY_Z, color="seagreen", linestyle="--", label=f"Long-spread entry (−{ENTRY_Z})")
ax_z.axhline(EXIT_Z, color="gray", linestyle=":", label=f"Exit level ({EXIT_Z})")
ax_z.scatter(zscore.index[long_entries], zscore.loc[long_entries], marker="^", color="limegreen", s=45, label="Buy pair / long spread", zorder=4)
ax_z.scatter(zscore.index[short_entries], zscore.loc[short_entries], marker="v", color="tomato", s=45, label="Sell pair / short spread", zorder=4)
ax_z.scatter(zscore.index[exits], zscore.loc[exits], marker="x", color="black", s=40, label="Exit position", zorder=4)
ax_z.set_title("3. Z-score signals and cumulative PnL vs. buy and hold")
ax_z.set_ylabel("Spread z-score")
ax_z.grid(alpha=0.25)

ax_pnl = ax_z.twinx()
ax_pnl.plot(pnl.index, pnl["cumulative_pnl"] * 100.0, color="purple", linewidth=1.8, label="Pairs strategy net PnL")
ax_pnl.plot(buy_hold.index, buy_hold * 100.0, color="saddlebrown", linewidth=1.5, label="50/50 buy-and-hold")
ax_pnl.set_ylabel("Cumulative return (%)")

z_lines, z_labels = ax_z.get_legend_handles_labels()
p_lines, p_labels = ax_pnl.get_legend_handles_labels()
ax_z.legend(z_lines + p_lines, z_labels + p_labels, loc="upper left", ncol=2, fontsize=8)
axes[2].set_xlabel("Date")
fig.tight_layout()
plt.show()


## Interpretation notes

- `βₜ` and the spread are estimated from the supplied historical price series. The strategy applies signals to the next session to avoid same-close look-ahead.
- The 50/50 benchmark is long both stocks and is not market-neutral; it is included as a simple reference, not a like-for-like risk benchmark.
- The backtest is sensitive to the date range, rolling window, entry/exit levels, hedge-ratio settings, and transaction-cost assumption. Use a chronological out-of-sample period before drawing conclusions.
